# 🔐 TrustVault AI
### Privacy-First Federated AI Assistant


| Section | Description |
|---|---|
| 0 | Setup & Installs |
| 1 | Configuration |
| 2 | NLP Models — Text Gen, Summarization, Q&A |
| 3 | Federated Learning Simulation |
| 4 | Differential Privacy (Opacus) |
| 5 | Homomorphic Encryption Simulation |
| 6 | RAG Pipeline (FAISS + Embeddings) |
| 7 | Streamlit App |
| 8 | Launch via Ngrok |



---
## Section 0 — Setup & Installs
Install all required packages.

In [1]:
# Install all dependencies
!pip install transformers accelerate sentencepiece --quiet
!pip install torch --quiet
!pip install opacus --quiet
!pip install faiss-cpu sentence-transformers --quiet
!pip install streamlit pyngrok --quiet
!pip install datasets --quiet

print("✅ All packages installed successfully.")

✅ All packages installed successfully.


In [2]:
# Core imports used throughout the notebook
import os
import json
import torch
import numpy as np
from pathlib import Path

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu (free-tier Colab — all models run fine on CPU)'}")

PyTorch version : 2.11.0+cu128
GPU available   : True
Device          : cuda


---
## Section 1 — Configuration


>
> - Get your Ngrok token from: https://dashboard.ngrok.com/get-started/your-authtoken
> - For privacy Set your secret keys.
> - Choose your own app username and password below.

In [3]:
# ============================================================
#  USER CONFIGURATION — Fill in your values here
# ============================================================

# Ngrok auth token (get yours at https://dashboard.ngrok.com)
from google.colab import userdata
NGROK_TOKEN = userdata.get('NGROK_TOKEN')

# App login credentials
APP_USERNAME = "admin"
APP_PASSWORD = "trustvault"

# Project working directory (all files will be created here)
PROJECT_DIR = Path("/content/trustvault")

# Number of simulated federated clients
NUM_CLIENTS = 3

# Differential Privacy settings
DP_NOISE_MULTIPLIER = 1.0   # Higher = more privacy, less accuracy
DP_MAX_GRAD_NORM   = 1.0
DP_EPSILON_TARGET  = 8.0    # Privacy budget

# ============================================================

# Create project folders
DATA_DIR    = PROJECT_DIR / "data"
MODELS_DIR  = PROJECT_DIR / "models"
FL_DIR      = PROJECT_DIR / "federated"
APP_DIR     = PROJECT_DIR / "app"

for d in [DATA_DIR, MODELS_DIR, FL_DIR, APP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Configuration loaded.")
print(f"   Project directory : {PROJECT_DIR}")
print(f"   FL clients        : {NUM_CLIENTS}")
print(f"   DP noise          : {DP_NOISE_MULTIPLIER}")

✅ Configuration loaded.
   Project directory : /content/trustvault
   FL clients        : 3
   DP noise          : 1.0


---
## Section 2 — NLP Models

TrustVault uses three lightweight transformer models, all run **locally** — no data is sent to any external server:

| Task | Model | Size |
|---|---|---|
| Text Generation | GPT-Neo 125M (EleutherAI) | ~500MB |
| Summarization | T5-Small (Google) | ~240MB |
| Question Answering | DistilBERT (HuggingFace) | ~260MB |

Models are loaded once and reused across all tasks.

In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    T5Tokenizer,
    T5ForConditionalGeneration,
    pipeline
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Text Generation: GPT-Neo 125M ─────────────────────────────
print("⏳ Loading GPT-Neo 125M (text generation)...")
gen_tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
gen_model     = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125M").to(DEVICE)
print("✅ GPT-Neo 125M loaded.")

# ── Summarization: T5-Small ───────────────────────────────────
print("⏳ Loading T5-Small (summarization)...")
sum_tokenizer = T5Tokenizer.from_pretrained("t5-small")
sum_model     = T5ForConditionalGeneration.from_pretrained("t5-small").to(DEVICE)
print("✅ T5-Small loaded.")

# ── Question Answering: DistilBERT ────────────────────────────
print("⏳ Loading DistilBERT (question answering)...")
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    device=0 if torch.cuda.is_available() else -1
)
print("✅ DistilBERT Q&A pipeline loaded.")

print("\n🎉 All models ready.")

⏳ Loading GPT-Neo 125M (text generation)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ GPT-Neo 125M loaded.
⏳ Loading T5-Small (summarization)...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

✅ T5-Small loaded.
⏳ Loading DistilBERT (question answering)...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

✅ DistilBERT Q&A pipeline loaded.

🎉 All models ready.


### 2.1 — Task Functions
Define clean, reusable functions for each NLP task.

In [5]:
# ── Text Generation ───────────────────────────────────────────
def generate_text(prompt: str, max_new_tokens: int = 100) -> str:
    """
    Generate text continuation from a prompt using GPT-Neo 125M.
    Runs entirely locally — no data sent externally.
    """
    inputs  = gen_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=gen_tokenizer.eos_token_id
    )
    return gen_tokenizer.decode(outputs[0], skip_special_tokens=True)


# ── Summarization ─────────────────────────────────────────────
def summarize_text(text: str, max_length: int = 120) -> str:
    """
    Summarize a long piece of text using T5-Small.
    Input is prefixed with 'summarize:' as required by T5.
    """
    input_text = "summarize: " + text.strip().replace("\n", " ")
    input_ids  = sum_tokenizer.encode(
        input_text, return_tensors="pt", truncation=True, max_length=512
    ).to(DEVICE)
    summary_ids = sum_model.generate(
        input_ids, max_length=max_length, num_beams=4, early_stopping=True
    )
    return sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# ── Question Answering ────────────────────────────────────────
def answer_question(context: str, question: str) -> dict:
    """
    Answer a question given a context passage using DistilBERT.
    Returns the answer string and confidence score.
    """
    if not context.strip():
        raise ValueError("Context cannot be empty.")
    if not question.strip():
        raise ValueError("Question cannot be empty.")
    result = qa_pipeline(question=question, context=context)
    return {"answer": result["answer"], "score": round(result["score"], 4)}


print("✅ NLP task functions defined.")

✅ NLP task functions defined.


### 2.2 — Quick Test
Verify all three models work before proceeding.

In [6]:
# ── Test: Text Generation ─────────────────────────────────────
prompt = "Privacy in AI systems is important because"
gen_output = generate_text(prompt, max_new_tokens=60)
print("📝 Text Generation Output:")
print(gen_output)
print()

# ── Test: Summarization ───────────────────────────────────────
long_text = """
Artificial intelligence is rapidly transforming industries by automating tasks,
improving decision-making, and enabling new capabilities. From healthcare to finance,
AI applications are enhancing productivity and delivering innovative solutions.
However, these advancements raise important concerns about data privacy, algorithmic
bias, and the need for transparent and explainable systems.
"""
summary = summarize_text(long_text)
print("📄 Summarization Output:")
print(summary)
print()

# ── Test: Q&A ─────────────────────────────────────────────────
context  = "Federated learning allows multiple clients to train a shared model on local data without sharing raw data with a central server."
question = "What does federated learning allow?"
qa_result = answer_question(context, question)
print("❓ Q&A Output:")
print(f"   Answer     : {qa_result['answer']}")
print(f"   Confidence : {qa_result['score']}")

📝 Text Generation Output:
Privacy in AI systems is important because it means that the user has the ability to manipulate data and is free to use the data as needed. In this case, the user’s data will be used for the intended purpose and the data will be modified accordingly.

A typical AI system consists of two main components: the first

📄 Summarization Output:
artificial intelligence is rapidly transforming industries by automating tasks, improving decision-making, and enabling new capabilities. these advancements raise concerns about data privacy, algorithmic bias, and the need for transparent and explainable systems.

❓ Q&A Output:
   Answer     : multiple clients to train a shared model on local data
   Confidence : 0.2994


---
## Section 3 — Federated Learning Simulation

**What is Federated Learning?**
Instead of sending raw user data to a central server, each client trains a model locally and sends only the **model weight updates** (gradients) to the server. The server aggregates these using **FedAvg** (Federated Averaging) to update the global model.

```
Client 1 ──┐
            ├──► Server (FedAvg) ──► Global Model
Client 2 ──┤          ↑
            │    Encrypted updates only
Client 3 ──┘    (no raw data)
```

We simulate this with 3 clients, each holding a different shard of the dataset.

In [7]:
# ── Step 1: Create a simple dataset and split across clients ──

# Sample Q&A dataset in JSONL format
dataset = [
    {"text": "Privacy in AI means protecting user data from unauthorized access.",   "label": 1},
    {"text": "Federated learning trains models without sharing raw data.",             "label": 1},
    {"text": "Differential privacy adds noise to protect individual data points.",    "label": 1},
    {"text": "Centralized servers store all user data in one location.",              "label": 0},
    {"text": "Cloud AI systems send user queries to remote servers.",                 "label": 0},
    {"text": "TrustVault runs all models locally on the user's device.",              "label": 1},
    {"text": "Model updates are encrypted before being sent to the server.",          "label": 1},
    {"text": "Data breaches expose sensitive user information to attackers.",         "label": 0},
    {"text": "Local inference ensures no personal data leaves the device.",           "label": 1},
    {"text": "Homomorphic encryption allows computation on encrypted data.",          "label": 1},
    {"text": "Open AI sends prompts to external APIs for processing.",               "label": 0},
    {"text": "Privacy-preserving ML protects users in sensitive domains.",            "label": 1},
]

# Save full dataset
dataset_path = DATA_DIR / "trustvault_dataset.jsonl"
with open(dataset_path, "w") as f:
    for item in dataset:
        f.write(json.dumps(item) + "\n")

# Split dataset into NUM_CLIENTS shards
chunk_size = len(dataset) // NUM_CLIENTS + 1
shards = [dataset[i:i+chunk_size] for i in range(0, len(dataset), chunk_size)]

for i, shard in enumerate(shards[:NUM_CLIENTS], start=1):
    shard_path = FL_DIR / f"client{i}.jsonl"
    with open(shard_path, "w") as f:
        for item in shard:
            f.write(json.dumps(item) + "\n")
    print(f"   Client {i}: {len(shard)} samples → {shard_path.name}")

print(f"\n✅ Dataset split into {NUM_CLIENTS} client shards.")

   Client 1: 5 samples → client1.jsonl
   Client 2: 5 samples → client2.jsonl
   Client 3: 2 samples → client3.jsonl

✅ Dataset split into 3 client shards.


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import copy

# ── Shared model: DistilBERT for binary classification ────────
FL_MODEL_NAME = "distilbert-base-uncased"

fl_tokenizer = AutoTokenizer.from_pretrained(FL_MODEL_NAME)

def get_fresh_model():
    """Return a fresh DistilBERT classification model."""
    return AutoModelForSequenceClassification.from_pretrained(
        FL_MODEL_NAME, num_labels=2
    ).to(DEVICE)


# ── Dataset class ─────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=64):
        self.encodings = tokenizer(
            [d["text"] for d in data],
            truncation=True, padding=True,
            max_length=max_len, return_tensors="pt"
        )
        self.labels = torch.tensor([d["label"] for d in data])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            {k: v[idx] for k, v in self.encodings.items()},
            self.labels[idx]
        )


# ── Client training function ──────────────────────────────────
def client_train(model, data, epochs=1, lr=2e-5):
    """
    Train a model on local client data for one round.
    Returns updated model weights — NOT the raw data.
    """
    dataset    = TextDataset(data, fl_tokenizer)
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion  = nn.CrossEntropyLoss()

    model.train()
    total_loss = 0
    for _ in range(epochs):
        for inputs, labels in dataloader:
            inputs  = {k: v.to(DEVICE) for k, v in inputs.items()}
            labels  = labels.to(DEVICE)
            outputs = model(**inputs)
            loss    = criterion(outputs.logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    return model.state_dict(), avg_loss


print("✅ Federated Learning components defined.")

✅ Federated Learning components defined.


In [9]:
# ── FedAvg: aggregate weights from all clients ─────────────────
def federated_average(client_weights: list) -> dict:
    """
    Federated Averaging (FedAvg) — McMahan et al. 2017.
    Averages model weights from all clients into a single global model.
    """
    avg_weights = copy.deepcopy(client_weights[0])
    for key in avg_weights:
        for i in range(1, len(client_weights)):
            avg_weights[key] += client_weights[i][key]
        avg_weights[key] = torch.div(avg_weights[key], len(client_weights))
    return avg_weights


# ── Run one full Federated Learning round ─────────────────────
print("🚀 Starting Federated Learning simulation...")
print(f"   Clients: {NUM_CLIENTS} | Device: {DEVICE}")
print("-" * 50)

# Load global model
global_model = get_fresh_model()

client_weights = []
for i in range(1, NUM_CLIENTS + 1):
    # Each client gets a copy of the global model
    local_model = copy.deepcopy(global_model)

    # Load client's local data
    shard_path  = FL_DIR / f"client{i}.jsonl"
    with open(shard_path) as f:
        local_data = [json.loads(line) for line in f]

    # Train locally
    weights, loss = client_train(local_model, local_data)
    client_weights.append(weights)
    print(f"   ✅ Client {i} trained | Local loss: {loss:.4f} | Data size: {len(local_data)}")

# Server aggregates via FedAvg
print("-" * 50)
print("🧠 Server aggregating with FedAvg...")
global_weights = federated_average(client_weights)
global_model.load_state_dict(global_weights)

# Save global model
global_model_path = MODELS_DIR / "global_model.pt"
torch.save(global_weights, global_model_path)
print(f"✅ Global model updated and saved → {global_model_path}")

🚀 Starting Federated Learning simulation...
   Clients: 3 | Device: cuda
--------------------------------------------------


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   ✅ Client 1 trained | Local loss: 0.6998 | Data size: 5
   ✅ Client 2 trained | Local loss: 0.7302 | Data size: 5
   ✅ Client 3 trained | Local loss: 0.6881 | Data size: 2
--------------------------------------------------
🧠 Server aggregating with FedAvg...
✅ Global model updated and saved → /content/trustvault/models/global_model.pt


---
## Section 4 — Differential Privacy

**What is Differential Privacy?**
Differential Privacy (DP) adds carefully calibrated **statistical noise** to model gradients during training. This ensures that no individual data point can be inferred from the model's output — even if an attacker has access to the model weights.

We use **Opacus** (Meta's PyTorch DP library) to apply DP during client training.

Key parameters:
- `noise_multiplier` — how much noise to add (higher = more private, less accurate)
- `max_grad_norm` — clips gradients before noise is added
- `epsilon (ε)` — privacy budget (lower = stronger privacy guarantee)

In [10]:
from opacus import PrivacyEngine, GradSampleModule
from opacus.accountants import RDPAccountant
from opacus.optimizers import DPOptimizer
from opacus.validators import ModuleValidator
import torch.nn as nn
from torch.utils.data import DataLoader

def client_train_with_dp(data, noise_multiplier=1.0, max_grad_norm=1.0, epochs=1, lr=2e-5):
    model = get_fresh_model()
    model = ModuleValidator.fix(model).to(DEVICE)

    for name, param in model.named_parameters():
        if 'embedding' in name or 'vocab' in name:
            param.requires_grad_(False)

    model.train()

    dataset    = TextDataset(data, fl_tokenizer)
    # Use full dataset as single batch — avoids all Opacus batch-size issues
    dataloader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    optimizer  = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion  = nn.CrossEntropyLoss()

    privacy_engine = PrivacyEngine()
    model, optimizer, dataloader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=dataloader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=max_grad_norm,
    )

    model.train()
    total_loss  = 0
    num_batches = 0
    for _ in range(epochs):
        for inputs, labels in dataloader:
            if labels.numel() == 0:
                continue
            inputs  = {k: v.to(DEVICE) for k, v in inputs.items()}
            labels  = labels.to(DEVICE)
            outputs = model(**inputs)
            loss    = criterion(outputs.logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss  += loss.item()
            num_batches += 1

    epsilon  = privacy_engine.get_epsilon(delta=1e-5)
    avg_loss = total_loss / max(num_batches, 1)
    return model._module.state_dict(), avg_loss, epsilon

print("✅ Differential Privacy training function defined.")

✅ Differential Privacy training function defined.


In [11]:
# ── Run FL with Differential Privacy ─────────────────────────
print("🔐 Running Federated Learning with Differential Privacy...")
print(f"   Noise multiplier : {DP_NOISE_MULTIPLIER}")
print(f"   Max grad norm    : {DP_MAX_GRAD_NORM}")
print("-" * 50)

dp_client_weights = []
for i in range(1, NUM_CLIENTS + 1):
    shard_path = FL_DIR / f"client{i}.jsonl"
    with open(shard_path) as f:
        local_data = [json.loads(line) for line in f]

    weights, loss, epsilon = client_train_with_dp(
        local_data,
        noise_multiplier=DP_NOISE_MULTIPLIER,
        max_grad_norm=DP_MAX_GRAD_NORM
    )
    dp_client_weights.append(weights)
    print(f"   ✅ Client {i} | Loss: {loss:.4f} | ε (epsilon): {epsilon:.4f}")

# Aggregate DP-trained weights
print("-" * 50)
dp_global_weights = federated_average(dp_client_weights)
dp_model_path = MODELS_DIR / "global_model_dp.pt"
torch.save(dp_global_weights, dp_model_path)
print(f"✅ DP-trained global model saved → {dp_model_path}")
print(f"\n📊 Privacy Budget (ε): Lower is stronger privacy.")
print(f"   Target ε : {DP_EPSILON_TARGET}")
print(f"   Achieved ε (last client): {epsilon:.4f}")

🔐 Running Federated Learning with Differential Privacy...
   Noise multiplier : 1.0
   Max grad norm    : 1.0
--------------------------------------------------


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retr

   ✅ Client 1 | Loss: 0.6814 | ε (epsilon): 4.3874


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   ✅ Client 2 | Loss: 0.6718 | ε (epsilon): 4.3874


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   ✅ Client 3 | Loss: 0.6427 | ε (epsilon): 4.3874
--------------------------------------------------
✅ DP-trained global model saved → /content/trustvault/models/global_model_dp.pt

📊 Privacy Budget (ε): Lower is stronger privacy.
   Target ε : 8.0
   Achieved ε (last client): 4.3874


---
## Section 5 — Homomorphic Encryption Simulation

**What is Homomorphic Encryption (HE)?**
HE allows computation on **encrypted data** without decrypting it first — the server can aggregate model updates without ever seeing the raw weights.

Full HE (e.g., TenSEAL/SEAL) requires significant compute and is impractical on Colab free tier. We simulate the **conceptual flow** — encrypt → aggregate → decrypt — using a lightweight mock cipher to demonstrate the architecture.

> In a production system, the `encrypt` and `decrypt` functions would be replaced with TenSEAL's CKKS scheme.

In [12]:
# ── Homomorphic Encryption Simulation ─────────────────────────
# Mock cipher: XOR with a key (demonstrates the encrypt/aggregate/decrypt flow)
# In production: replace with TenSEAL CKKS scheme

HE_KEY = 42  # Symmetric key (shared between clients and server)

def he_encrypt(value: float) -> float:
    """
    Simulate encrypting a scalar value.
    Production: use TenSEAL CKKS for real float encryption.
    """
    import struct, math
    # Pack float → int bits → XOR with key → unpack back to float
    int_bits = struct.unpack('I', struct.pack('f', value))[0]
    encrypted = int_bits ^ (HE_KEY << 10)  # Shift key to affect mantissa bits
    try:
        result = struct.unpack('f', struct.pack('I', encrypted))[0]
        return result if math.isfinite(result) else value + HE_KEY * 0.001
    except:
        return value + HE_KEY * 0.001

def he_decrypt(value: float) -> float:
    """Decrypt a simulated encrypted value (XOR is its own inverse)."""
    return he_encrypt(value)  # XOR encryption is symmetric

def encrypt_weights(state_dict: dict) -> dict:
    """Encrypt all model weight tensors (simulated)."""
    encrypted = {}
    for key, tensor in state_dict.items():
        flat = tensor.cpu().float().numpy().flatten()
        enc_flat = np.array([he_encrypt(v) for v in flat], dtype=np.float32)
        encrypted[key] = enc_flat.reshape(tensor.shape)
    return encrypted

def decrypt_weights(encrypted_dict: dict, ref_state_dict: dict) -> dict:
    """Decrypt all model weight tensors and restore to PyTorch tensors."""
    decrypted = {}
    for key, enc_arr in encrypted_dict.items():
        flat = enc_arr.flatten()
        dec_flat = np.array([he_decrypt(v) for v in flat], dtype=np.float32)
        decrypted[key] = torch.tensor(
            dec_flat.reshape(enc_arr.shape),
            dtype=ref_state_dict[key].dtype
        )
    return decrypted


print("✅ Homomorphic Encryption simulation defined.")

✅ Homomorphic Encryption simulation defined.


In [13]:
# ── Demonstrate the full HE flow on a small example ───────────
print("🔐 Demonstrating Homomorphic Encryption flow...")
print("-" * 50)

# Use a tiny model to keep this fast
from transformers import DistilBertConfig, DistilBertForSequenceClassification

tiny_config = DistilBertConfig(n_layers=1, n_heads=2, dim=32, hidden_dim=64, num_labels=2)
tiny_model  = DistilBertForSequenceClassification(tiny_config)
ref_weights = tiny_model.state_dict()

# Pick one layer to demonstrate
demo_key   = list(ref_weights.keys())[0]
orig_value = ref_weights[demo_key].float().numpy().flatten()[0]

# Encrypt
enc_weights = encrypt_weights({demo_key: ref_weights[demo_key]})
enc_value   = enc_weights[demo_key].flatten()[0]

# Decrypt
dec_weights = decrypt_weights(enc_weights, {demo_key: ref_weights[demo_key]})
dec_value   = dec_weights[demo_key].float().numpy().flatten()[0]

print(f"   Layer          : {demo_key}")
print(f"   Original value : {orig_value:.6f}")
print(f"   Encrypted value: {enc_value:.6f}  ← looks random to the server")
print(f"   Decrypted value: {dec_value:.6f}")
print(f"   Match          : {'✅ Yes' if abs(orig_value - dec_value) < 1e-3 else '⚠️ Small rounding error (expected in simulation)'}")
print("-" * 50)
print("✅ HE simulation complete. In production, replace with TenSEAL CKKS.")

🔐 Demonstrating Homomorphic Encryption flow...
--------------------------------------------------
   Layer          : distilbert.embeddings.word_embeddings.weight
   Original value : 0.000000
   Encrypted value: 0.000000  ← looks random to the server
   Decrypted value: 0.000000
   Match          : ✅ Yes
--------------------------------------------------
✅ HE simulation complete. In production, replace with TenSEAL CKKS.


---
## Section 6 — RAG Pipeline

**What is RAG (Retrieval-Augmented Generation)?**
Instead of relying solely on a model's pre-trained knowledge, RAG:
1. Converts a knowledge base into dense vector embeddings
2. Stores them in a FAISS vector index
3. At query time, retrieves the most relevant passages
4. Feeds them as context into the Q&A model

This makes answers more accurate and grounded in your specific data — all without any external API calls.

In [14]:
import faiss
from sentence_transformers import SentenceTransformer

# ── Load embedding model ──────────────────────────────────────
print("⏳ Loading sentence embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded.")


# ── Knowledge base ─────────────────────────────────────────────
# In a real deployment this would be loaded from files
KNOWLEDGE_BASE = [
    "Federated learning allows multiple clients to collaboratively train a model without sharing raw data.",
    "Differential privacy protects individual data by adding statistical noise to model updates.",
    "Homomorphic encryption enables computation on encrypted data without decryption.",
    "TrustVault uses GPT-Neo for text generation, T5-Small for summarization, and DistilBERT for Q&A.",
    "The FedAvg algorithm averages model weights from all participating clients each round.",
    "Opacus is a PyTorch library by Meta that enables differential privacy in training.",
    "FAISS is a library by Meta for efficient similarity search on dense vector embeddings.",
    "Local inference means all computation happens on the user's device with no cloud dependency.",
    "Privacy budget (epsilon) measures how much information about individual data can be leaked.",
    "Sentence transformers convert text into dense vector representations for semantic search.",
]

# ── Build FAISS index ─────────────────────────────────────────
print("⏳ Building FAISS vector index...")
embeddings = embed_model.encode(KNOWLEDGE_BASE, convert_to_numpy=True)
dimension  = embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)  # L2 distance index
faiss_index.add(embeddings)

print(f"✅ FAISS index built.")
print(f"   Passages indexed : {faiss_index.ntotal}")
print(f"   Embedding dim    : {dimension}")

⏳ Loading sentence embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded.
⏳ Building FAISS vector index...
✅ FAISS index built.
   Passages indexed : 10
   Embedding dim    : 384


In [15]:
# ── RAG retrieval + Q&A function ──────────────────────────────
def rag_answer(query: str, top_k: int = 3) -> dict:
    """
    Retrieve the top_k most relevant passages from the knowledge base
    and answer the query using DistilBERT Q&A.
    Returns the answer, confidence score, and retrieved context.
    """
    # Embed the query
    query_embedding = embed_model.encode([query], convert_to_numpy=True)

    # Search FAISS index
    distances, indices = faiss_index.search(query_embedding, top_k)

    # Build context from retrieved passages
    retrieved = [KNOWLEDGE_BASE[i] for i in indices[0] if i < len(KNOWLEDGE_BASE)]
    context   = " ".join(retrieved)

    # Answer using DistilBERT Q&A
    result = qa_pipeline(question=query, context=context)

    return {
        "answer"    : result["answer"],
        "score"     : round(result["score"], 4),
        "context"   : context,
        "retrieved" : retrieved
    }


# ── Test RAG ──────────────────────────────────────────────────
test_query = "How does differential privacy protect user data?"
rag_result = rag_answer(test_query)

print("🔍 RAG Pipeline Test")
print("-" * 50)
print(f"Query          : {test_query}")
print(f"Answer         : {rag_result['answer']}")
print(f"Confidence     : {rag_result['score']}")
print(f"\nRetrieved passages:")
for i, p in enumerate(rag_result['retrieved'], 1):
    print(f"  [{i}] {p}")

🔍 RAG Pipeline Test
--------------------------------------------------
Query          : How does differential privacy protect user data?
Answer         : by adding statistical noise to model updates
Confidence     : 0.2738

Retrieved passages:
  [1] Differential privacy protects individual data by adding statistical noise to model updates.
  [2] Privacy budget (epsilon) measures how much information about individual data can be leaked.
  [3] Opacus is a PyTorch library by Meta that enables differential privacy in training.


---
## Section 7 — Streamlit App

Write the complete Streamlit app to disk. The app provides:
- 🔐 Login authentication
- 🛡️ Privacy mode toggle
- 📝 Text Generation tab
- 📄 Summarization tab (with file upload)
- ❓ Q&A tab (standard + RAG mode)
- 📊 Privacy dashboard (shows FL round and ε spent)

In [16]:
app_code = r'''
import streamlit as st
import torch
import sys
import os

st.set_page_config(page_title="TrustVault AI", page_icon="🔐", layout="wide")

APP_USERNAME = "admin"
APP_PASSWORD = "trustvault"

if "authenticated" not in st.session_state:
    st.session_state.authenticated = False

if not st.session_state.authenticated:
    st.title("🔐 TrustVault AI")
    st.subheader("Login")
    username = st.text_input("Username")
    password = st.text_input("Password", type="password")
    if st.button("Login"):
        if username == APP_USERNAME and password == APP_PASSWORD:
            st.session_state.authenticated = True
            st.rerun()
        else:
            st.error("❌ Invalid credentials")
    st.stop()

st.sidebar.title("🔐 TrustVault AI")
st.sidebar.markdown("**Privacy-First Federated AI Assistant**")
st.sidebar.divider()

privacy_mode = st.sidebar.toggle("🛡️ Privacy Mode", value=True)
if privacy_mode:
    st.sidebar.success("Privacy Mode ON — No data stored or transmitted")
else:
    st.sidebar.warning("Privacy Mode OFF")

st.sidebar.divider()
mode = st.sidebar.radio(
    "Select Task",
    ["📝 Text Generation", "📄 Summarization", "❓ Q&A", "🔍 RAG Q&A", "📊 Privacy Dashboard"]
)

if st.sidebar.button("Logout"):
    st.session_state.authenticated = False
    st.rerun()

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    T5Tokenizer, T5ForConditionalGeneration,
    pipeline
)
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

@st.cache_resource
def load_gen_model():
    tok = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
    mdl = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125M").to(DEVICE)
    return tok, mdl

@st.cache_resource
def load_sum_model():
    tok = T5Tokenizer.from_pretrained("t5-small")
    mdl = T5ForConditionalGeneration.from_pretrained("t5-small").to(DEVICE)
    return tok, mdl

@st.cache_resource
def load_qa_model():
    return pipeline("question-answering", model="distilbert-base-cased-distilled-squad",
                    device=0 if torch.cuda.is_available() else -1)

@st.cache_resource
def load_rag():
    embed = SentenceTransformer("all-MiniLM-L6-v2")
    kb = [
        "Federated learning allows multiple clients to collaboratively train a model without sharing raw data.",
        "Differential privacy protects individual data by adding statistical noise to model updates.",
        "Homomorphic encryption enables computation on encrypted data without decryption.",
        "TrustVault uses GPT-Neo for text generation, T5-Small for summarization, and DistilBERT for Q&A.",
        "The FedAvg algorithm averages model weights from all participating clients each round.",
        "Opacus is a PyTorch library by Meta that enables differential privacy in training.",
        "FAISS is a library by Meta for efficient similarity search on dense vector embeddings.",
        "Local inference means all computation happens on the user device with no cloud dependency.",
        "Privacy budget epsilon measures how much information about individual data can be leaked.",
        "Sentence transformers convert text into dense vector representations for semantic search.",
    ]
    emb = embed.encode(kb, convert_to_numpy=True)
    idx = faiss.IndexFlatL2(emb.shape[1])
    idx.add(emb)
    return embed, idx, kb

if mode == "📝 Text Generation":
    st.title("📝 Text Generation")
    st.caption("Powered by GPT-Neo 125M — runs locally")
    prompt = st.text_area("Enter your prompt:", height=120,
                          placeholder="Privacy in AI systems is important because...")
    max_tokens = st.slider("Max new tokens", 50, 200, 100)
    if st.button("Generate"):
        if not prompt.strip():
            st.warning("Please enter a prompt.")
        else:
            with st.spinner("Generating..."):
                tok, mdl = load_gen_model()
                inputs  = tok(prompt, return_tensors="pt").to(DEVICE)
                outputs = mdl.generate(**inputs, max_new_tokens=max_tokens,
                                       do_sample=True, top_k=50, top_p=0.95,
                                       temperature=0.8,
                                       pad_token_id=tok.eos_token_id)
                result = tok.decode(outputs[0], skip_special_tokens=True)
            st.subheader("Generated Output:")
            st.write(result)
            if privacy_mode:
                st.info("🛡️ Privacy Mode: Input not stored or transmitted.")

elif mode == "📄 Summarization":
    st.title("📄 Text Summarization")
    st.caption("Powered by T5-Small — runs locally")
    uploaded = st.file_uploader("Upload a .txt file (optional)", type=["txt"])
    if uploaded:
        text_input = uploaded.read().decode("utf-8")
        st.text_area("Loaded text:", text_input, height=150)
    else:
        text_input = st.text_area("Or paste text here:", height=200,
                                  placeholder="Paste a long article or paragraph...")
    if st.button("Summarize"):
        if not text_input.strip():
            st.warning("Please provide some text.")
        else:
            with st.spinner("Summarizing..."):
                tok, mdl = load_sum_model()
                inp = "summarize: " + text_input.strip().replace("\n", " ")
                ids = tok.encode(inp, return_tensors="pt",
                                 truncation=True, max_length=512).to(DEVICE)
                out = mdl.generate(ids, max_length=150, num_beams=4, early_stopping=True)
                summary = tok.decode(out[0], skip_special_tokens=True)
            st.subheader("Summary:")
            st.success(summary)

elif mode == "❓ Q&A":
    st.title("❓ Question Answering")
    st.caption("Powered by DistilBERT — runs locally")
    context  = st.text_area("Context (paste a paragraph):", height=180)
    question = st.text_input("Your question:")
    if st.button("Answer"):
        if not context.strip() or not question.strip():
            st.warning("Please provide both context and a question.")
        else:
            with st.spinner("Finding answer..."):
                qa = load_qa_model()
                result = qa(question=question, context=context)
            st.subheader("Answer:")
            st.success(result["answer"])
            st.caption(f"Confidence: {result['score']:.2%}")

elif mode == "🔍 RAG Q&A":
    st.title("🔍 RAG Question Answering")
    st.caption("Retrieval-Augmented Generation using FAISS + DistilBERT")
    st.info("Uses the built-in TrustVault knowledge base. Ask anything about privacy, federated learning, or this project.")
    question = st.text_input("Ask a question:")
    top_k    = st.slider("Passages to retrieve", 1, 5, 3)
    if st.button("Search & Answer"):
        if not question.strip():
            st.warning("Please enter a question.")
        else:
            with st.spinner("Retrieving and answering..."):
                embed, idx, kb = load_rag()
                qa = load_qa_model()
                q_emb = embed.encode([question], convert_to_numpy=True)
                _, indices = idx.search(q_emb, top_k)
                retrieved = [kb[i] for i in indices[0] if i < len(kb)]
                context   = " ".join(retrieved)
                result    = qa(question=question, context=context)
            st.subheader("Answer:")
            st.success(result["answer"])
            st.caption(f"Confidence: {result['score']:.2%}")
            with st.expander("Retrieved passages"):
                for i, p in enumerate(retrieved, 1):
                    st.write(f"**[{i}]** {p}")

elif mode == "📊 Privacy Dashboard":
    st.title("📊 Privacy Dashboard")
    col1, col2, col3 = st.columns(3)
    col1.metric("FL Clients",       "3",    "Simulated")
    col2.metric("DP Noise (σ)",    "1.0",   "Opacus")
    col3.metric("Privacy Budget ε", "≤8.0", "Lower = stronger")
    st.divider()
    st.subheader("Privacy Techniques Active")
    st.markdown("""
    | Technique | Status | Library |
    |---|---|---|
    | Federated Learning | ✅ Simulated (3 clients, FedAvg) | Custom |
    | Differential Privacy | ✅ Active | Opacus |
    | Homomorphic Encryption | 🔄 Simulated (mock cipher) | Custom |
    | Local Inference | ✅ All models run on-device | HuggingFace |
    | RAG (no external API) | ✅ FAISS + local embeddings | FAISS |
    """)
    st.divider()
    st.caption("TrustVault AI — KIIT University B.Tech Final Year Project")
'''

app_path = APP_DIR / "app.py"
with open(app_path, "w") as f:
    f.write(app_code.strip())

print(f"✅ Streamlit app written → {app_path}")

✅ Streamlit app written → /content/trustvault/app/app.py


---
## Section 8 — Launch via Ngrok

Starts the Streamlit app and creates a public URL using Ngrok.

> ⚠️ Make sure you filled in `NGROK_TOKEN` in **Section 1** before running this cell.
>
> The public URL will be printed below — open it in any browser.

In [17]:
import subprocess
import time
from pyngrok import ngrok

# Kill any existing Streamlit or Ngrok processes
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
try:
    ngrok.kill()
except:
    pass
time.sleep(1)

# Validate token
if NGROK_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    raise ValueError(
        "❌ You haven't set your Ngrok token.\n"
        "   Go to https://dashboard.ngrok.com/get-started/your-authtoken\n"
        "   and paste your token into NGROK_TOKEN in Section 1."
    )

# Set Ngrok auth token
ngrok.set_auth_token(NGROK_TOKEN)

# Start Streamlit in the background
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", str(app_path),
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)  # Wait for Streamlit to start

# Create Ngrok tunnel
public_url = ngrok.connect(addr=8501, proto="http")

print("🚀 TrustVault AI is live!")
print("-" * 50)
print(f"🔗 Public URL : {public_url}")
print(f"👤 Username   : {APP_USERNAME}")
print(f"🔑 Password   : {APP_PASSWORD}")
print("-" * 50)
print("ℹ️  To stop: Runtime → Interrupt execution")

🚀 TrustVault AI is live!
--------------------------------------------------
🔗 Public URL : NgrokTunnel: "https://sterility-glamorous-vanquish.ngrok-free.dev" -> "http://localhost:8501"
👤 Username   : admin
🔑 Password   : trustvault
--------------------------------------------------
ℹ️  To stop: Runtime → Interrupt execution


---
## ✅ Notebook Complete

| Section | What was built |
|---|---|
| 0 | Installed all dependencies |
| 1 | Configured project settings in one place |
| 2 | Loaded GPT-Neo, T5-Small, DistilBERT locally |
| 3 | Simulated Federated Learning with FedAvg across 3 clients |
| 4 | Applied Differential Privacy via Opacus |
| 5 | Demonstrated Homomorphic Encryption simulation |
| 6 | Built RAG pipeline with FAISS + sentence embeddings |
| 7 | Wrote full Streamlit app with login, privacy toggle, all tasks |
| 8 | Launched publicly via Ngrok |

---

**References:**
- McMahan et al. (2017) — [Communication-Efficient Learning of Deep Networks from Decentralized Data](https://arxiv.org/abs/1602.05629)
- Dwork (2006) — [Differential Privacy](https://www.microsoft.com/en-us/research/publication/differential-privacy/)
- [Opacus](https://opacus.ai/) — PyTorch Differential Privacy
- [HuggingFace Transformers](https://huggingface.co/docs/transformers)
- [FAISS](https://github.com/facebookresearch/faiss) — Meta AI